# Music Annotation Dataset Forensics

**Runtime:** CPU. Select **Runtime > Run all**.

This standalone notebook performs read-only forensic analysis of the V3 training dataset. It does not change masks, tiles, labels or models.

It answers:

- How much of each tile is labelled annotation?
- Are positive tiles genuinely useful or only barely positive?
- Which mask tiles produce no usable YOLO polygon?
- Do stored mask tiles exactly match crops from their source training masks?
- Do image, mask and label files pair correctly?
- How fragmented are the annotation masks?
- How many annotations touch tile edges and may be truncated?
- Are some pages dominating the dataset?
- Are training and validation distributions materially different?
- Which tiles have the least, most, or most fragmented annotation coverage?
- Which tiles are probable noise, probable weak signal, or probable label-generation defects?

Outputs include CSV and JSON data, summary charts, contact sheets and automatic recommendations.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Configuration

The notebook reads the existing V3 configuration where available. No generated data is overwritten.

In [ ]:
import cv2, json, csv, hashlib, re, math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
CONFIG_PATH=ROOT/'project_config.json'
CONFIG=json.loads(CONFIG_PATH.read_text()) if CONFIG_PATH.exists() else {}
DATASET_DIR=ROOT/'dataset'
TRAINING_MASK_DIR=ROOT/'masks'/'training'
MANIFEST_PATH=ROOT/'dataset_manifest_v3.json'
OUTPUT_ROOT=ROOT/'dataset_forensics'
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUTPUT_DIR=OUTPUT_ROOT/RUN_NAME
CHART_DIR=OUTPUT_DIR/'charts'
CONTACT_DIR=OUTPUT_DIR/'contact_sheets'
TABLE_DIR=OUTPUT_DIR/'tables'
for folder in [OUTPUT_DIR,CHART_DIR,CONTACT_DIR,TABLE_DIR]: folder.mkdir(parents=True,exist_ok=True)

TILE_SIZE=int(CONFIG.get('tile_size',640))
MIN_CONTOUR_AREA=float(CONFIG.get('labels',{}).get('minimum_contour_area',12))
POLYGON_EPSILON=float(CONFIG.get('labels',{}).get('polygon_epsilon_fraction',0.002))

# Recommendation thresholds. These classify tiles for review; they do not modify data.
BARELY_POSITIVE_PIXELS=25
LOW_COVERAGE_PCT=0.05
HIGH_COVERAGE_PCT=15.0
EXTREME_COVERAGE_PCT=35.0
HIGH_FRAGMENTATION_COMPONENTS=100
DOMINANT_PAGE_SHARE_PCT=15.0
EDGE_MARGIN_PIXELS=2
MAX_CONTACT_SHEET_TILES=20

print('Dataset:',DATASET_DIR)
print('Output:',OUTPUT_DIR)
print('Tile size:',TILE_SIZE)
print('Minimum contour area:',MIN_CONTOUR_AREA)

## Dataset discovery and file-pair validation

In [ ]:
SUPPORTED_IMAGES={'.jpg','.jpeg','.png','.tif','.tiff'}

def image_files(folder):
    folder=Path(folder)
    return sorted([p for p in folder.iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED_IMAGES]) if folder.exists() else []

def parse_tile_name(stem):
    match=re.match(r'^(.*)_y(\d+)_x(\d+)$',stem)
    if not match: return None,None,None
    return match.group(1),int(match.group(2)),int(match.group(3))

def file_md5(path):
    h=hashlib.md5()
    with open(path,'rb') as f:
        for block in iter(lambda:f.read(1024*1024),b''): h.update(block)
    return h.hexdigest()

pairing_issues=[]
split_files={}
for split in ['train','val']:
    image_dir=DATASET_DIR/'images'/split
    mask_dir=DATASET_DIR/'mask_tiles'/split
    label_dir=DATASET_DIR/'labels'/split
    images={p.stem:p for p in image_files(image_dir)}
    masks={p.stem:p for p in image_files(mask_dir)}
    labels={p.stem:p for p in label_dir.glob('*.txt')} if label_dir.exists() else {}
    all_stems=sorted(set(images)|set(masks)|set(labels))
    for stem in all_stems:
        missing=[]
        if stem not in images: missing.append('image')
        if stem not in masks: missing.append('mask')
        if stem not in labels: missing.append('label')
        if missing: pairing_issues.append({'split':split,'tile':stem,'missing':','.join(missing)})
    split_files[split]={'images':images,'masks':masks,'labels':labels}
    print(split,'images',len(images),'masks',len(masks),'labels',len(labels))

print('Pairing issues:',len(pairing_issues))
if pairing_issues: display(pd.DataFrame(pairing_issues).head(50))

## Analyse each mask tile and YOLO label

The analysis includes pixel coverage, connected-component geometry, contour survival, fragmentation, border contact, polygon counts, and source-crop consistency.

In [ ]:
def read_yolo_label(path):
    lines=[]; coordinate_problems=0
    if path is None or not path.exists(): return lines,coordinate_problems
    for raw in path.read_text().splitlines():
        if not raw.strip(): continue
        parts=raw.split()
        try:
            coords=[float(v) for v in parts[1:]]
            valid=(parts[0]=='0' and len(coords)>=6 and len(coords)%2==0 and all(0<=v<=1 for v in coords))
            if not valid: coordinate_problems+=1
            lines.append(coords)
        except Exception:
            coordinate_problems+=1
    return lines,coordinate_problems

def analyse_mask(mask):
    binary=np.where(mask>0,255,0).astype(np.uint8)
    white_pixels=int(np.sum(binary>0))
    coverage_pct=100*white_pixels/binary.size
    n,labels,stats,_=cv2.connectedComponentsWithStats(binary,8)
    component_areas=[int(stats[i,cv2.CC_STAT_AREA]) for i in range(1,n)]
    contours,_=cv2.findContours(binary,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE)
    contour_areas=[float(cv2.contourArea(c)) for c in contours]
    valid_contours=[]
    valid_polygons=[]
    for contour in contours:
        area=cv2.contourArea(contour)
        if area<MIN_CONTOUR_AREA: continue
        valid_contours.append(contour)
        polygon=cv2.approxPolyDP(contour,max(1.0,POLYGON_EPSILON*cv2.arcLength(contour,True)),True)
        if len(polygon)>=3: valid_polygons.append(polygon)
    border=np.zeros_like(binary)
    border[:EDGE_MARGIN_PIXELS,:]=255; border[-EDGE_MARGIN_PIXELS:,:]=255; border[:,:EDGE_MARGIN_PIXELS]=255; border[:,-EDGE_MARGIN_PIXELS:]=255
    edge_pixels=int(np.sum((binary>0)&(border>0)))
    edge_touch_components=0
    for i in range(1,n):
        comp=labels==i
        if np.any(comp&(border>0)): edge_touch_components+=1
    return {
        'white_pixels':white_pixels,'annotation_pct':coverage_pct,'component_count':len(component_areas),
        'largest_component':max(component_areas) if component_areas else 0,
        'median_component':float(np.median(component_areas)) if component_areas else 0,
        'tiny_components_lt_6':sum(a<6 for a in component_areas),
        'contour_count':len(contours),'valid_contour_count':len(valid_contours),'generated_polygon_count':len(valid_polygons),
        'largest_contour_area':max(contour_areas) if contour_areas else 0,
        'edge_annotation_pixels':edge_pixels,'edge_touch_components':edge_touch_components,
        'fragmentation_ratio':len(component_areas)/max(white_pixels,1)
    }

rows=[]
for split in ['train','val']:
    fileset=split_files[split]
    for stem,mask_path in sorted(fileset['masks'].items()):
        mask=cv2.imread(str(mask_path),cv2.IMREAD_GRAYSCALE)
        if mask is None: continue
        page,y,x=parse_tile_name(stem)
        stats=analyse_mask(mask)
        yolo_lines,coordinate_problems=read_yolo_label(fileset['labels'].get(stem))
        source_match=None
        source_path=TRAINING_MASK_DIR/f'{page}.png' if page else None
        if source_path and source_path.exists() and y is not None:
            source=cv2.imread(str(source_path),cv2.IMREAD_GRAYSCALE)
            fresh=source[y:y+mask.shape[0],x:x+mask.shape[1]]
            source_match=bool(fresh.shape==mask.shape and np.array_equal(fresh,mask))
        row={'tile':stem,'page':page,'split':split,'mask_path':str(mask_path),'image_path':str(fileset['images'].get(stem,'')),'label_path':str(fileset['labels'].get(stem,'')),'source_crop_matches':source_match,'label_line_count':len(yolo_lines),'label_coordinate_problems':coordinate_problems}
        row.update(stats)
        row['pixel_positive']=row['white_pixels']>0
        row['polygon_positive']=row['generated_polygon_count']>0
        rows.append(row)

df=pd.DataFrame(rows)
print('Tiles analysed:',len(df))
print('Positive by pixels:',int(df.pixel_positive.sum()))
print('Positive by polygons:',int(df.polygon_positive.sum()))
print('Source-crop mismatches:',int((df.source_crop_matches==False).sum()))
print('Coordinate problems:',int(df.label_coordinate_problems.sum()))

## Automatic issue classification and recommendations

Recommendations are intentionally conservative. The notebook never downgrades or deletes a tile automatically.

In [ ]:
def recommend(row):
    issues=[]
    if row['source_crop_matches'] is False: issues.append('CRITICAL: stored mask differs from source crop')
    if row['label_coordinate_problems']>0: issues.append('CRITICAL: invalid YOLO coordinates')
    if row['white_pixels']==0 and row['label_line_count']>0: issues.append('CRITICAL: labels exist for empty mask')
    if row['white_pixels']>0 and row['generated_polygon_count']==0: issues.append('REVIEW: positive mask produces no polygon')
    if 0<row['white_pixels']<BARELY_POSITIVE_PIXELS: issues.append('REVIEW: barely positive, probable speck/remnant')
    if 0<row['annotation_pct']<LOW_COVERAGE_PCT: issues.append('REVIEW: extremely low annotation coverage')
    if row['annotation_pct']>EXTREME_COVERAGE_PCT: issues.append('REVIEW: extreme coverage, possible over-expanded or erroneous mask')
    elif row['annotation_pct']>HIGH_COVERAGE_PCT: issues.append('CHECK: unusually high coverage')
    if row['component_count']>HIGH_FRAGMENTATION_COMPONENTS: issues.append('REVIEW: highly fragmented mask')
    if row['edge_touch_components']>0: issues.append('CHECK: annotation touches tile edge')
    if row['label_line_count']!=row['generated_polygon_count']: issues.append('CHECK: saved label count differs from current polygon count')
    return ' | '.join(issues) if issues else 'OK'

df['recommendation']=df.apply(recommend,axis=1)
df['severity']=df.recommendation.map(lambda x:'critical' if 'CRITICAL' in x else ('review' if 'REVIEW' in x else ('check' if 'CHECK' in x else 'ok')))
print(df.severity.value_counts())
display(df[df.severity!='ok'][['tile','split','white_pixels','annotation_pct','component_count','generated_polygon_count','label_line_count','recommendation']].head(100))

## Summary statistics and split comparison

In [ ]:
positive=df[df.white_pixels>0].copy()
metrics=['white_pixels','annotation_pct','component_count','largest_component','median_component','valid_contour_count','generated_polygon_count','edge_touch_components','fragmentation_ratio']
print('ALL POSITIVE TILES')
display(positive[metrics].describe(percentiles=[.01,.05,.10,.25,.5,.75,.9,.95,.99]).T)
print('BY SPLIT')
display(positive.groupby('split')[metrics].agg(['count','mean','median','min','max']))
page_summary=df.groupby(['split','page']).agg(tiles=('tile','count'),positive_tiles=('pixel_positive','sum'),annotation_pixels=('white_pixels','sum'),mean_coverage_pct=('annotation_pct','mean'),median_coverage_pct=('annotation_pct','median'),polygons=('generated_polygon_count','sum'),review_tiles=('severity',lambda s:int((s!='ok').sum()))).reset_index()
page_summary['positive_tile_pct']=100*page_summary.positive_tiles/page_summary.tiles
page_summary['dataset_tile_share_pct']=100*page_summary.tiles/len(df)
display(page_summary)
if (page_summary.dataset_tile_share_pct>DOMINANT_PAGE_SHARE_PCT).any():
    print('WARNING: one or more pages exceed',DOMINANT_PAGE_SHARE_PCT,'% of all tiles.')

## Distribution charts

In [ ]:
def save_hist(column,title,xlabel,logx=False):
    values=positive[column].replace([np.inf,-np.inf],np.nan).dropna()
    plt.figure(figsize=(11,5)); plt.hist(values,bins=60); plt.title(title); plt.xlabel(xlabel); plt.ylabel('Positive tiles')
    if logx: plt.xscale('log')
    plt.tight_layout(); path=CHART_DIR/f'{column}_histogram.png'; plt.savefig(path,dpi=160); plt.show()

save_hist('annotation_pct','Annotation coverage across positive tiles','Annotation coverage (%)')
save_hist('white_pixels','White annotation pixels across positive tiles','White pixels',True)
save_hist('component_count','Connected components per positive tile','Components',True)
save_hist('largest_component','Largest connected component per positive tile','Pixels',True)

plt.figure(figsize=(10,6));
for split,group in positive.groupby('split'):
    plt.scatter(group.annotation_pct,group.generated_polygon_count,s=12,alpha=.45,label=split)
plt.xlabel('Annotation coverage (%)');plt.ylabel('Generated polygon count');plt.title('Coverage versus polygon count');plt.legend();plt.tight_layout();plt.savefig(CHART_DIR/'coverage_vs_polygons.png',dpi=160);plt.show()

plt.figure(figsize=(12,6)); page_summary.sort_values('annotation_pixels',ascending=False).plot.bar(x='page',y='annotation_pixels',legend=False,ax=plt.gca());plt.ylabel('Annotation pixels');plt.title('Annotation-pixel contribution by page');plt.tight_layout();plt.savefig(CHART_DIR/'annotation_pixels_by_page.png',dpi=160);plt.show()

## Contact sheets

The notebook creates review sheets for the most informative categories: lowest coverage, highest coverage, highest fragmentation, no-polygon failures, edge-touching annotations, and largest components. Each panel displays the image tile beside its mask.

In [ ]:
def contact_sheet(subset,title,file_name,limit=MAX_CONTACT_SHEET_TILES):
    subset=subset.head(limit)
    if subset.empty: print('No tiles for',title); return
    rows=len(subset); fig,axes=plt.subplots(rows,2,figsize=(14,max(4,rows*4)))
    if rows==1: axes=np.array([axes])
    for axis_row,(_,row) in zip(axes,subset.iterrows()):
        image=cv2.imread(row.image_path); mask=cv2.imread(row.mask_path,0)
        if image is not None: axis_row[0].imshow(cv2.cvtColor(image,cv2.COLOR_BGR2RGB))
        axis_row[0].set_title(f"{row.tile}\ncoverage={row.annotation_pct:.4f}% polygons={row.generated_polygon_count}")
        axis_row[1].imshow(mask,cmap='gray')
        axis_row[1].set_title(f"pixels={row.white_pixels} comps={row.component_count} edge={row.edge_touch_components}")
        for a in axis_row:a.axis('off')
    fig.suptitle(title);plt.tight_layout();path=CONTACT_DIR/file_name;plt.savefig(path,dpi=140,bbox_inches='tight');plt.show()

contact_sheet(positive.sort_values('annotation_pct'),'Lowest annotation coverage','lowest_coverage.png')
contact_sheet(positive.sort_values('annotation_pct',ascending=False),'Highest annotation coverage','highest_coverage.png')
contact_sheet(positive.sort_values('component_count',ascending=False),'Most fragmented masks','most_fragmented.png')
contact_sheet(positive[positive.generated_polygon_count==0].sort_values('white_pixels',ascending=False),'Positive masks with no polygons','no_polygon_failures.png')
contact_sheet(positive[positive.edge_touch_components>0].sort_values('edge_annotation_pixels',ascending=False),'Annotations touching tile edges','edge_touching.png')
contact_sheet(positive.sort_values('largest_component',ascending=False),'Largest annotation components','largest_components.png')

## Threshold sensitivity simulation

This simulation estimates how changing minimum contour area would alter the number of valid polygons and no-polygon positive tiles. It does not rewrite labels.

In [ ]:
thresholds=[1,3,6,12,20,30,50,100]
sensitivity=[]
for threshold in thresholds:
    zero_tiles=0;polygon_total=0
    for _,row in positive.iterrows():
        mask=cv2.imread(row.mask_path,0);contours,_=cv2.findContours(mask,cv2.RETR_EXTERNAL,cv2.CHAIN_APPROX_SIMPLE);count=sum(cv2.contourArea(c)>=threshold for c in contours);polygon_total+=count
        if count==0:zero_tiles+=1
    sensitivity.append({'minimum_contour_area':threshold,'total_surviving_contours':polygon_total,'positive_tiles_with_zero_contours':zero_tiles})
sensitivity_df=pd.DataFrame(sensitivity);display(sensitivity_df)
plt.figure(figsize=(10,5));plt.plot(sensitivity_df.minimum_contour_area,sensitivity_df.positive_tiles_with_zero_contours,marker='o');plt.xlabel('Minimum contour area');plt.ylabel('Positive tiles with zero surviving contours');plt.title('Contour threshold sensitivity');plt.grid(True);plt.tight_layout();plt.savefig(CHART_DIR/'contour_threshold_sensitivity.png',dpi=160);plt.show()

## Export CSV, JSON and executive summary

In [ ]:
tile_csv=TABLE_DIR/'tile_forensics.csv'; page_csv=TABLE_DIR/'page_summary.csv'; sensitivity_csv=TABLE_DIR/'contour_sensitivity.csv'; summary_json=OUTPUT_DIR/'forensics_summary.json'; report_txt=OUTPUT_DIR/'executive_summary.txt'
df.to_csv(tile_csv,index=False);page_summary.to_csv(page_csv,index=False);sensitivity_df.to_csv(sensitivity_csv,index=False)
summary={
 'created_utc':datetime.now(timezone.utc).isoformat(),'dataset_dir':str(DATASET_DIR),'tile_size':TILE_SIZE,'minimum_contour_area':MIN_CONTOUR_AREA,
 'tiles_total':int(len(df)),'positive_tiles':int(df.pixel_positive.sum()),'negative_tiles':int((~df.pixel_positive).sum()),
 'positive_tiles_no_polygon':int(((df.pixel_positive)&(~df.polygon_positive)).sum()),'source_crop_mismatches':int((df.source_crop_matches==False).sum()),
 'label_coordinate_problems':int(df.label_coordinate_problems.sum()),'severity_counts':{k:int(v) for k,v in df.severity.value_counts().items()},
 'positive_coverage_percentiles':{str(q):float(positive.annotation_pct.quantile(q)) for q in [.01,.05,.1,.25,.5,.75,.9,.95,.99]},
 'highest_page_tile_share_pct':float(page_summary.dataset_tile_share_pct.max()),'outputs':{'tile_csv':str(tile_csv),'page_csv':str(page_csv),'sensitivity_csv':str(sensitivity_csv),'charts':str(CHART_DIR),'contact_sheets':str(CONTACT_DIR)}
}
summary_json.write_text(json.dumps(summary,indent=2))
recommendations=[]
if summary['source_crop_mismatches']>0:recommendations.append('STOP: stored mask tiles do not all match source training-mask crops.')
if summary['label_coordinate_problems']>0:recommendations.append('STOP: one or more YOLO labels contain invalid coordinates or formats.')
if summary['positive_tiles_no_polygon']>0:recommendations.append('Review positive tiles with no polygon before silently downgrading them.')
median_cov=summary['positive_coverage_percentiles']['0.5']
if median_cov<0.1:recommendations.append('Median annotation coverage is below 0.1%; consider smaller or annotation-centred tiles, not only more dilation.')
if float(page_summary.dataset_tile_share_pct.max())>DOMINANT_PAGE_SHARE_PCT:recommendations.append('One page contributes a disproportionate share of tiles; consider page-balanced sampling.')
if not recommendations:recommendations.append('No critical structural defect was found. Continue with visual review of the generated contact sheets.')
report='DATASET FORENSICS EXECUTIVE SUMMARY\n\n'+json.dumps(summary,indent=2)+'\n\nRECOMMENDATIONS\n- '+'\n- '.join(recommendations)
report_txt.write_text(report)
print(report)
print('\nTile CSV:',tile_csv)
print('Page CSV:',page_csv)
print('Sensitivity CSV:',sensitivity_csv)
print('Charts:',CHART_DIR)
print('Contact sheets:',CONTACT_DIR)
print('JSON summary:',summary_json)